In [2]:
# Importing essential Libraries for project 

import pandas as pd 
import numpy as np 
import re
import random 
from datetime import datetime,timedelta

np.random.seed(42)
random.seed(42)
print("Libraries Downloaded successfully")

Libraries Downloaded successfully


# Generating Raw messy data of real estate hiring

In [3]:
num_rows = 3200
sources = ['Naukri', 'LinkedIn', 'Indeed', 'JobHai', 'Career Page', 'Apna', 'WorkIndia', 'Walk-In', 'Referral']
source_weights = [0.30, 0.18, 0.14, 0.10, 0.08, 0.06, 0.05, 0.05, 0.04]

roles = ['Sales Executive', 'Pre-Sales', 'Sales Manager', 'HR Recruiter', 'VP Sales']
role_weights = [0.45, 0.30, 0.15, 0.07, 0.03]

markets = ['Gurgaon', 'Noida', 'Dubai']
recruiters = ['Sachin', 'Sameer', 'Shriyanshi', 'Deepanshi', 'Anu', 'Chetan']

names_pool = [
    'Rahul Sharma', 'Amit Verma', 'Priya Singh', 'Neha Gupta', 'Rohit Yadav', 
    'Ankit Malhotra', 'Pooja Kapoor', 'Vikas Chauhan', 'Simran Jain', 'Karan Agarwal', 
    'Deepak Mehta', 'Sneha Bansal', 'Manish Tyagi', 'Riya Rana', 'Saurabh Nair', 
    'Nisha Dubey', 'Abhishek Tiwari', 'Komal Joshi', 'Tarun Bhatia', 'Divya Saxena'
]

statuses = [
    'Not Looking for Job', 'Not Interested', 'Not Suitable',
    'Rejected in HR Interview', 'F2F Scheduled - No Show',
    'Rejected in Final Interview', 'Shortlisted', 'Offer Declined', 'Joined'
]
status_weights = [0.23, 0.18, 0.15, 0.16, 0.12, 0.08, 0.04, 0.03, 0.01]

remarks_pool = {
    'Not Looking for Job': ['already joined somewhere', 'not looking for job change currently', 'happy in current org'],
    'Not Interested': ['not interested in real estate', 'wants WFH / desk job only', 'location too far - Noida/Gurgaon issue'],
    'Not Suitable': ['fresher - no exp', 'no real estate experience', 'from BPO background, not suitable', 'overage / irrelevant profile'],
    'Rejected in HR Interview': ['bad communication skills', 'lacks confidence in pitching', 'salary expectation 45% higher than budget', 'poor attitude during screening'],
    'F2F Scheduled - No Show': ['confirmed for F2F but did not visit', 'phone switched off on interview day', 'rescheduled twice, no show'],
    'Rejected in Final Interview': ['weak closing skills in final round', 'sales manager rejected - low target clarity', 'lack of luxury/Dubai market knowledge'],
    'Shortlisted': ['selected in final round, offer approval pending', 'docs submitted, release in progress'],
    'Offer Declined': ['declined offer - got counter offer from current company', 'fixed salary too low', 'family not agreeing for relocation'],
    'Joined': ['joined on time, induction done', 'onboarded for sales team']
}

raw_data = []
base_date = datetime(2026, 8, 1)

for i in range(1, num_rows + 1):
    src = np.random.choice(sources, p=source_weights)
    role = np.random.choice(roles, p=role_weights)
    market = np.random.choice(markets, p=[0.45, 0.35, 0.20])
    rec = random.choice(recruiters)
    status = np.random.choice(statuses, p=status_weights)
    
    cand_name = random.choice(names_pool)
    base_phone = str(random.randint(7000000000, 9999999999))
    
    if i % 15 == 0 and i > 20:
        cand_name = raw_data[i - 10]['name']
        base_phone = raw_data[i - 10]['phone_number'][-10:]
        
    phone_formats = [base_phone, f"+91 {base_phone}", f"0{base_phone}", f"{base_phone[:5]}-{base_phone[5:]}"]
    messy_phone = random.choice(phone_formats)
    
    applied_dt = base_date + timedelta(days=random.randint(0, 90))
    call_delay = random.choices([0, 1, 2, 4, 6], weights=[0.35, 0.30, 0.15, 0.12, 0.08])[0]
    contacted_dt = applied_dt + timedelta(days=call_delay)
    
    remark_text = f"[{contacted_dt.strftime('%d-%b')}] {random.choice(remarks_pool[status])}"
    
    raw_data.append({
        's_no': i,
        'name': cand_name,
        'phone_number': messy_phone,
        'source': src,
        'role': role,
        'target_market': market,
        'contacted_by': rec,
        'applied_date': applied_dt.strftime('%Y-%m-%d'),
        'contacted_date': contacted_dt.strftime('%Y-%m-%d'),
        'current_status': status,
        'remarks': remark_text
    })

df_raw = pd.DataFrame(raw_data)

raw_file_path = r"C:\Users\himsh\OneDrive\Desktop\talent-funnel-bottleneck-analytics\data\raw_hiring_tracker.csv"
df_raw.to_csv(raw_file_path, index=False)

print("Raw data saved:", df_raw.shape)

Raw data saved: (3200, 11)


In [4]:
df_raw.head(5)

,s_no,name,phone_number,source,role,target_market,contacted_by,applied_date,contacted_date,current_status,remarks
0,1,Neha Gupta,07107420369,LinkedIn,HR Recruiter,Noida,Chetan,2026-09-01,2026-09-01,Rejected in HR Interview,[01-Sep] bad communication skills
1,2,Komal Joshi,73733-99426,Naukri,Sales Executive,Gurgaon,Chetan,2026-08-05,2026-08-05,Rejected in Final Interview,[05-Aug] weak closing skills in final round
2,3,Abhishek Tiwari,9585650756,Indeed,Pre-Sales,Gurgaon,Sameer,2026-10-11,2026-10-11,Offer Declined,[11-Oct] family not agreeing for relocation
3,4,Komal Joshi,+91 8801823908,Apna,Sales Executive,Gurgaon,Chetan,2026-09-27,2026-09-28,Not Looking for Job,[28-Sep] already joined somewhere
4,5,Riya Rana,08461364854,LinkedIn,Pre-Sales,Gurgaon,Sameer,2026-08-20,2026-08-20,Not Interested,[20-Aug] wants WFH / desk job only


In [5]:
df = df_raw.copy()
df.head()

,s_no,name,phone_number,source,role,target_market,contacted_by,applied_date,contacted_date,current_status,remarks
0,1,Neha Gupta,07107420369,LinkedIn,HR Recruiter,Noida,Chetan,2026-09-01,2026-09-01,Rejected in HR Interview,[01-Sep] bad communication skills
1,2,Komal Joshi,73733-99426,Naukri,Sales Executive,Gurgaon,Chetan,2026-08-05,2026-08-05,Rejected in Final Interview,[05-Aug] weak closing skills in final round
2,3,Abhishek Tiwari,9585650756,Indeed,Pre-Sales,Gurgaon,Sameer,2026-10-11,2026-10-11,Offer Declined,[11-Oct] family not agreeing for relocation
3,4,Komal Joshi,+91 8801823908,Apna,Sales Executive,Gurgaon,Chetan,2026-09-27,2026-09-28,Not Looking for Job,[28-Sep] already joined somewhere
4,5,Riya Rana,08461364854,LinkedIn,Pre-Sales,Gurgaon,Sameer,2026-08-20,2026-08-20,Not Interested,[20-Aug] wants WFH / desk job only


In [6]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 3200 entries, 0 to 3199
Data columns (total 11 columns):
 #   Column          Non-Null Count  Dtype
---  ------          --------------  -----
 0   s_no            3200 non-null   int64
 1   name            3200 non-null   str  
 2   phone_number    3200 non-null   str  
 3   source          3200 non-null   str  
 4   role            3200 non-null   str  
 5   target_market   3200 non-null   str  
 6   contacted_by    3200 non-null   str  
 7   applied_date    3200 non-null   str  
 8   contacted_date  3200 non-null   str  
 9   current_status  3200 non-null   str  
 10  remarks         3200 non-null   str  
dtypes: int64(1), str(10)
memory usage: 689.0 KB


In [7]:
# finding duplicate values by phone_number
df[df['phone_number'].duplicated(keep=False)].sort_values('phone_number')

,s_no,name,phone_number,source,role,target_market,contacted_by,applied_date,contacted_date,current_status,remarks
2735,2736,Deepak Mehta,+91 7297394609,JobHai,Pre-Sales,Gurgaon,Shriyanshi,2026-08-01,2026-08-01,Not Interested,[01-Aug] wants WFH / desk job only
2744,2745,Deepak Mehta,+91 7297394609,Walk-In,Pre-Sales,Noida,Deepanshi,2026-08-27,2026-08-29,Rejected in HR Interview,[29-Aug] poor attitude during screening
1685,1686,Neha Gupta,+91 7473301871,Naukri,Sales Executive,Noida,Deepanshi,2026-08-03,2026-08-07,Rejected in Final Interview,[07-Aug] weak closing skills in final round
1694,1695,Neha Gupta,+91 7473301871,Career Page,Pre-Sales,Dubai,Shriyanshi,2026-08-27,2026-08-27,Not Looking for Job,[27-Aug] not looking for job change currently
1460,1461,Amit Verma,+91 7553260471,LinkedIn,Sales Executive,Gurgaon,Sachin,2026-10-23,2026-10-23,Not Looking for Job,[23-Oct] happy in current org
...,...,...,...,...,...,...,...,...,...,...,...
1964,1965,Karan Agarwal,9731524909,LinkedIn,Sales Executive,Gurgaon,Shriyanshi,2026-10-17,2026-10-18,Not Looking for Job,[18-Oct] already joined somewhere
1925,1926,Deepak Mehta,9782629027,Referral,Sales Manager,Noida,Shriyanshi,2026-08-24,2026-08-28,Not Interested,[28-Aug] wants WFH / desk job only
1934,1935,Deepak Mehta,9782629027,Walk-In,Sales Executive,Noida,Sameer,2026-09-01,2026-09-01,Not Looking for Job,[01-Sep] happy in current org
2900,2901,Rahul Sharma,9813438925,LinkedIn,Sales Manager,Dubai,Chetan,2026-08-09,2026-08-10,F2F Scheduled - No Show,[10-Aug] phone switched off on interview day


In [8]:
# 1. Pehle har row ke remark ke aage uska contacted person add karte hain
df['tagged_remark'] = df['contacted_by'] + ": " + df['remarks'].astype(str)

# 2. Phone number ke basis par merge (aggregate) karte hain
df = df.groupby('phone_number', as_index=False).agg({
    's_no': 'first',
    'name': 'first',
    'source': 'first',
    'role': 'first',
    'target_market': 'first',
    'applied_date': 'first',
    'contacted_date': 'first',
    'contacted_by': 'last',
    'current_status': 'last',
    'tagged_remark': ' | '.join
}).rename(columns={'tagged_remark': 'remarks'})

# 3. Columns ko wapas original order mein set karna
cols_order = ['s_no', 'name', 'phone_number', 'source', 'role', 'target_market', 
              'contacted_by', 'applied_date', 'contacted_date', 'current_status', 'remarks']
df = df[cols_order]

print("Merge ke baad total unique candidates:", len(df))
df[df['remarks'].str.contains(r'\|')].head()

Merge ke baad total unique candidates: 3163


,s_no,name,phone_number,source,role,target_market,contacted_by,applied_date,contacted_date,current_status,remarks
81,2736,Deepak Mehta,+91 7297394609,JobHai,Pre-Sales,Gurgaon,Deepanshi,2026-08-01,2026-08-01,Rejected in HR Interview,Shriyanshi: [01-Aug] wants WFH / desk job only...
124,1686,Neha Gupta,+91 7473301871,Naukri,Sales Executive,Noida,Shriyanshi,2026-08-03,2026-08-07,Not Looking for Job,Deepanshi: [07-Aug] weak closing skills in fin...
144,1461,Amit Verma,+91 7553260471,LinkedIn,Sales Executive,Gurgaon,Deepanshi,2026-10-23,2026-10-23,Not Suitable,Sachin: [23-Oct] happy in current org | Deepan...
150,1731,Nisha Dubey,+91 7585845375,Walk-In,Sales Executive,Gurgaon,Deepanshi,2026-10-26,2026-10-27,Shortlisted,Anu: [27-Oct] lacks confidence in pitching | D...
225,486,Nisha Dubey,+91 7839332926,Naukri,Sales Executive,Noida,Sameer,2026-08-20,2026-08-20,Rejected in Final Interview,Anu: [20-Aug] poor attitude during screening |...


In [13]:
df['phone_number'] = df['phone_number'].astype(str).str.replace(r'\D', '', regex=True).str[-10:]
df = df.sort_values('s_no').reset_index(drop=True)
df['s_no'] = df.index + 1

df.head(10)

,s_no,name,phone_number,source,role,target_market,contacted_by,applied_date,contacted_date,current_status,remarks
0,1,Neha Gupta,7107420369,LinkedIn,HR Recruiter,Noida,Chetan,2026-09-01,2026-09-01,Rejected in HR Interview,Chetan: [01-Sep] bad communication skills
1,2,Komal Joshi,7373399426,Naukri,Sales Executive,Gurgaon,Chetan,2026-08-05,2026-08-05,Rejected in Final Interview,Chetan: [05-Aug] weak closing skills in final ...
2,3,Abhishek Tiwari,9585650756,Indeed,Pre-Sales,Gurgaon,Sameer,2026-10-11,2026-10-11,Offer Declined,Sameer: [11-Oct] family not agreeing for reloc...
3,4,Komal Joshi,8801823908,Apna,Sales Executive,Gurgaon,Chetan,2026-09-27,2026-09-28,Not Looking for Job,Chetan: [28-Sep] already joined somewhere
4,5,Riya Rana,8461364854,LinkedIn,Pre-Sales,Gurgaon,Sameer,2026-08-20,2026-08-20,Not Interested,Sameer: [20-Aug] wants WFH / desk job only
5,6,Priya Singh,8631775357,Indeed,Sales Executive,Gurgaon,Sachin,2026-09-15,2026-09-19,Not Interested,Sachin: [19-Sep] location too far - Noida/Gurg...
6,7,Amit Verma,8973214822,LinkedIn,Sales Manager,Gurgaon,Shriyanshi,2026-09-18,2026-09-18,Not Suitable,"Shriyanshi: [18-Sep] from BPO background, not ..."
7,8,Divya Saxena,8553210608,Indeed,Sales Executive,Noida,Chetan,2026-10-30,2026-10-30,Not Looking for Job,Chetan: [30-Oct] happy in current org
8,9,Karan Agarwal,7342703921,Naukri,HR Recruiter,Dubai,Sameer,2026-08-13,2026-08-14,F2F Scheduled - No Show,Sameer: [14-Aug] phone switched off on intervi...
9,10,Sneha Bansal,7698594025,LinkedIn,Sales Executive,Noida,Chetan,2026-09-15,2026-09-15,Not Suitable,"Chetan: [15-Sep] from BPO background, not suit..."


> All Cleaning is done , now save clean data and save it for further analysis

In [14]:
clean_file_path = r"C:\Users\himsh\OneDrive\Desktop\talent-funnel-bottleneck-analytics\data\clean_hiring_tracker.csv"
df.to_csv(clean_file_path, index=False)

print("Clean file saved successfully! Total rows:", len(df))

Clean file saved successfully! Total rows: 3163
